# Assignment 3 — Machine Learning Pipeline (Improved)

Written by Abdullah Ariyanto (z5543164).

## Project Objectives
* **Task 1 (Regression):** Predict `safety_rating` (Target: $RMSE \le 1$).
* **Task 2 (Classification):** Predict `claim` (Target: $F1 \ge 0.60$).

## Key Improvements Over Baseline
1. **Deep feature engineering** — parse `torque`, `power`, `car_age`, and the `features` list into numeric components
2. **Binary indicators** for each safety feature (critical for `safety_rating`)
3. **Physics-based interaction features** (power-to-weight, torque-to-weight, etc.)
4. **LightGBM + CatBoost ensemble** for regression
5. **Threshold-tuned classification** with SMOTE oversampling

## 1. Data Exploration, Cleaning & Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

display(train_df.info())
display(train_df.head())

# Check for missing values
missing = train_df.isnull().sum()
print("\nMissing values:")
display(missing[missing > 0])
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression Target Distribution
sns.histplot(train_df['safety_rating'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Safety Rating (Regression Target)')
axes[0].set_xlabel('Safety Rating')

# Classification Target Distribution
sns.countplot(data=train_df, x='claim', ax=axes[1], hue='claim', palette='Set2', legend=False)
axes[1].set_title('Distribution of Claims (Classification Target)')

plt.tight_layout()
plt.show()

print(f"Safety rating range: [{train_df['safety_rating'].min()}, {train_df['safety_rating'].max()}]")
print(f"\nClass balance for 'claim':")
print(train_df['claim'].value_counts(normalize=True))

### 1.1 Inspecting String Columns

Several columns encode rich numeric data as strings. We need to parse these to unlock their predictive power:
- `torque`: e.g., `"91Nm@4250rpm"` → two numbers (Nm, RPM)
- `power`: e.g., `"67.06bhp@5500rpm"` → two numbers (BHP, RPM)
- `car_age`: e.g., `"4 years and 6 months"` → single number (months)
- `features`: e.g., `"['esc', 'tpms', ...]"` → count + binary indicators for each safety feature

In [ ]:
print("Sample torque values:")
print(train_df['torque'].unique()[:10])
print(f"\nSample power values:")
print(train_df['power'].unique()[:10])
print(f"\nSample car_age values:")
print(train_df['car_age'].unique()[:10])
print(f"\nSample features values:")
print(train_df['features'].iloc[0])
print(train_df['features'].iloc[1])

## 2. Feature Engineering & Feature Selection

### 2.1 Feature Engineering Pipeline

The `features` column is critical — it lists safety equipment (ESC, TPMS, brake assist, etc.) that **directly determines** the vehicle's safety rating. A car with 16 safety features and 6 airbags will have a much higher safety rating than one with 3 features and 2 airbags.

We create:
1. **Parsed numerics**: `torque_nm`, `torque_rpm`, `power_bhp`, `power_rpm`, `car_age_months`
2. **Feature count**: `n_features` (number of safety/convenience features)
3. **Binary safety indicators**: `has_esc`, `has_tpms`, `has_brake_assist`, etc.
4. **Interaction features**: `power_to_weight`, `torque_to_weight`, `safety_density`, etc.

In [ ]:
# Key safety features found in the 'features' column
SAFETY_FEATURES = [
    'esc', 'tpms', 'brake_assist', 'parking_camera',
    'parking_sensors', 'front_fog_lights', 'adjustable_steering',
    'rear_defogger', 'power_door_locks', 'central_locking',
    'power_steering', 'driver_seat_adjustable', 'day_night_mirror',
    'ecw', 'speed_alert', 'rear_wiper', 'rear_washer'
]

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering — applied independently to train and test.
    No target information is used, so no data leakage.
    """
    d = df.copy()

    # 1. Parse torque: "91Nm@4250rpm" → torque_nm, torque_rpm
    if 'torque' in d.columns:
        ts = d['torque'].astype(str)
        d['torque_nm'] = pd.to_numeric(ts.str.extract(r'([\d.]+)\s*Nm', flags=re.IGNORECASE)[0], errors='coerce')
        d['torque_rpm'] = pd.to_numeric(ts.str.extract(r'@\s*([\d.]+)\s*rpm', flags=re.IGNORECASE)[0], errors='coerce')
        d.drop(columns=['torque'], inplace=True)

    # 2. Parse power: "67.06bhp@5500rpm" → power_bhp, power_rpm
    if 'power' in d.columns:
        ps = d['power'].astype(str)
        d['power_bhp'] = pd.to_numeric(ps.str.extract(r'([\d.]+)\s*bhp', flags=re.IGNORECASE)[0], errors='coerce')
        d['power_rpm'] = pd.to_numeric(ps.str.extract(r'@\s*([\d.]+)\s*rpm', flags=re.IGNORECASE)[0], errors='coerce')
        d.drop(columns=['power'], inplace=True)

    # 3. Parse car_age: "4 years and 6 months" → 54 (months)
    if 'car_age' in d.columns:
        age_s = d['car_age'].astype(str)
        yrs = pd.to_numeric(age_s.str.extract(r'(\d+)\s*year')[0], errors='coerce').fillna(0)
        mos = pd.to_numeric(age_s.str.extract(r'(\d+)\s*month')[0], errors='coerce').fillna(0)
        d['car_age_months'] = (yrs * 12 + mos).astype(int)
        d.drop(columns=['car_age'], inplace=True)

    # 4. Parse features list → count + binary indicators (MOST CRITICAL STEP)
    if 'features' in d.columns:
        fs = d['features'].astype(str)
        d['n_features'] = fs.str.count("'") // 2
        for feat in SAFETY_FEATURES:
            d[f'has_{feat}'] = fs.str.contains(feat, case=False, na=False).astype(int)
        d.drop(columns=['features'], inplace=True)

    # 5. Interaction features
    if 'power_bhp' in d.columns and 'gross_weight' in d.columns:
        d['power_to_weight'] = d['power_bhp'] / d['gross_weight'].replace(0, np.nan)
    if 'torque_nm' in d.columns and 'gross_weight' in d.columns:
        d['torque_to_weight'] = d['torque_nm'] / d['gross_weight'].replace(0, np.nan)
    if 'displacement' in d.columns and 'cylinder' in d.columns:
        d['disp_per_cylinder'] = d['displacement'] / d['cylinder'].replace(0, np.nan)
    if all(c in d.columns for c in ['length', 'width', 'height']):
        d['volume'] = d['length'] * d['width'] * d['height']
    if 'airbags' in d.columns and 'n_features' in d.columns:
        d['safety_density'] = d['airbags'] * d['n_features']
    if 'car_age_months' in d.columns and 'annual_mileage_km' in d.columns:
        d['est_total_mileage'] = d['car_age_months'] * d['annual_mileage_km'] / 12.0

    return d

print("engineer_features() defined.")

In [ ]:
train_clean = engineer_features(train_df)
test_clean = engineer_features(test_df)

print(f"Before feature engineering: {train_df.shape[1]} columns")
print(f"After feature engineering:  {train_clean.shape[1]} columns")
print(f"\nNew columns added:")
new_cols = set(train_clean.columns) - set(train_df.columns)
print(sorted(new_cols))

### 2.2 Validating the Feature Engineering

Let's verify the parsed features correlate strongly with `safety_rating`.

In [ ]:
# Correlation of new engineered features with safety_rating
eng_cols = ['n_features', 'safety_density', 'airbags', 'torque_nm', 'power_bhp',
            'car_age_months', 'power_to_weight', 'torque_to_weight',
            'displacement', 'gross_weight'] + [f'has_{f}' for f in SAFETY_FEATURES]

eng_cols_exist = [c for c in eng_cols if c in train_clean.columns]
corr = train_clean[eng_cols_exist + ['safety_rating']].corr()['safety_rating'].drop('safety_rating').sort_values(ascending=False)

print("Top correlations with safety_rating:")
display(corr.head(20))

fig, ax = plt.subplots(figsize=(10, 8))
corr.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Correlations with safety_rating')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

### 2.3 Missing Value Handling & Categorical Encoding

We use training-set medians for numeric imputation and `OrdinalEncoder` for remaining categorical columns.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# --- Missing values (use training stats for both) ---
exclude_targets = ['safety_rating', 'claim', 'policy_id']
num_cols = [c for c in train_clean.select_dtypes(include=['number']).columns if c not in exclude_targets]
medians = train_clean[num_cols].median()
train_clean[num_cols] = train_clean[num_cols].fillna(medians)
test_clean[num_cols] = test_clean[num_cols].fillna(medians)

cat_cols = [c for c in train_clean.select_dtypes(exclude=['number']).columns if c != 'policy_id']
if cat_cols:
    train_clean[cat_cols] = train_clean[cat_cols].fillna('Missing')
    test_clean[cat_cols] = test_clean[cat_cols].fillna('Missing')

# --- Ordinal encoding ---
if cat_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_clean[cat_cols] = encoder.fit_transform(train_clean[cat_cols])
    test_clean[cat_cols] = encoder.transform(test_clean[cat_cols])

print(f"Categorical columns encoded: {cat_cols}")
print(f"Numeric columns: {len(num_cols)}")
print(f"\nFinal train shape: {train_clean.shape}")
print(f"Final test shape:  {test_clean.shape}")

## 3. Regression Task: Predict Safety Rating

### 3.1 Train–Validation Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

train_final = train_clean.dropna(subset=['safety_rating', 'claim'])

base_features = [c for c in train_final.columns if c not in ['policy_id', 'safety_rating', 'claim']]

X_reg = train_final[base_features]
y_reg = train_final['safety_rating']

X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Number of features: {len(base_features)}")
print(f"Training set:   {X_train_reg.shape}")
print(f"Validation set: {X_val_reg.shape}")

### 3.2 Baseline Model Comparison

We compare **LightGBM**, **XGBoost**, and **CatBoost** with default parameters to establish a baseline.

In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

models_reg = {
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    "CatBoost": CatBoostRegressor(random_seed=42, verbose=0)
}

results_reg = []
for name, model in models_reg.items():
    start = time.time()
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_val_reg)
    rmse = root_mean_squared_error(y_val_reg, pred)
    elapsed = time.time() - start
    results_reg.append({"Model": name, "Validation RMSE": rmse, "Time (s)": round(elapsed, 2)})
    print(f"{name:12s} → RMSE: {rmse:.4f}  ({elapsed:.1f}s)")

results_reg_df = pd.DataFrame(results_reg)
display(results_reg_df)

### 3.3 Hyperparameter Tuning

We tune both LightGBM and CatBoost individually, then combine them in an ensemble.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

lgbm_param_dist = {
    'n_estimators': stats.randint(500, 3000),
    'learning_rate': stats.uniform(0.01, 0.09),
    'max_depth': [-1, 8, 10, 12, 15],
    'num_leaves': stats.randint(31, 200),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0, 0.01, 0.1, 1.0],
    'min_child_samples': [5, 10, 20, 30]
}

lgbm_search = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=lgbm_param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    verbose=1
)

print("Tuning LightGBM...")
lgbm_search.fit(X_train_reg, y_train_reg)

best_lgbm = lgbm_search.best_estimator_
lgbm_pred = best_lgbm.predict(X_val_reg)
lgbm_rmse = root_mean_squared_error(y_val_reg, lgbm_pred)
print(f"\nBest LightGBM params: {lgbm_search.best_params_}")
print(f"Tuned LightGBM RMSE: {lgbm_rmse:.4f}")

In [ ]:
# CatBoost tuning — manual grid since CatBoost doesn't play as well with sklearn search
best_cat_rmse = 999
best_cat_params = {}

cat_configs = [
    {'iterations': 1500, 'learning_rate': 0.03, 'depth': 8, 'l2_leaf_reg': 3},
    {'iterations': 2000, 'learning_rate': 0.03, 'depth': 8, 'l2_leaf_reg': 1},
    {'iterations': 2000, 'learning_rate': 0.02, 'depth': 10, 'l2_leaf_reg': 3},
    {'iterations': 1500, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 5},
    {'iterations': 2500, 'learning_rate': 0.02, 'depth': 8, 'l2_leaf_reg': 1},
]

print("Tuning CatBoost...")
for i, cfg in enumerate(cat_configs):
    m = CatBoostRegressor(**cfg, random_seed=42, verbose=0)
    m.fit(X_train_reg, y_train_reg)
    p = m.predict(X_val_reg)
    rmse_val = root_mean_squared_error(y_val_reg, p)
    print(f"  Config {i+1}: RMSE={rmse_val:.4f} | {cfg}")
    if rmse_val < best_cat_rmse:
        best_cat_rmse = rmse_val
        best_cat_params = cfg
        best_cat_model = m

cat_pred = best_cat_model.predict(X_val_reg)
print(f"\nBest CatBoost RMSE: {best_cat_rmse:.4f}")
print(f"Best CatBoost params: {best_cat_params}")

### 3.4 Ensemble Evaluation

Averaging predictions from different model families often reduces variance and improves generalization.

In [ ]:
# Try different ensemble weights
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens_pred = w * lgbm_pred + (1 - w) * cat_pred
    ens_rmse = root_mean_squared_error(y_val_reg, ens_pred)
    print(f"Ensemble (LGBM={w:.1f}, CatBoost={1-w:.1f}): RMSE = {ens_rmse:.4f}")

# Pick the best weight (or default to 0.5)
best_w = 0.5
best_ens_rmse = 999
for w in np.arange(0.2, 0.8, 0.05):
    ens_pred = w * lgbm_pred + (1 - w) * cat_pred
    ens_rmse = root_mean_squared_error(y_val_reg, ens_pred)
    if ens_rmse < best_ens_rmse:
        best_ens_rmse = ens_rmse
        best_w = w

print(f"\nOptimal weight: LGBM={best_w:.2f}, CatBoost={1-best_w:.2f}")
print(f"Best ensemble RMSE: {best_ens_rmse:.4f}")

### 3.5 Feature Importance Analysis

Let's examine which features the models rely on most.

In [ ]:
fi = pd.DataFrame({
    'feature': base_features,
    'importance': best_lgbm.feature_importances_
}).sort_values('importance', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi, x='importance', y='feature', ax=ax, color='steelblue')
ax.set_title('Top 25 Features — LightGBM Regression')
plt.tight_layout()
plt.show()

## 4. Classification Task: Predict Claim

### 4.1 Train–Validation Split (stratified)

Class imbalance is severe (~93.8% no claim). We use stratification and class weighting.

In [ ]:
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

clf_features = base_features + ['safety_rating']  # use true safety_rating from training as feature

X_clf = train_final[clf_features]
y_clf = train_final['claim']

X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"Classification Training: {X_train_clf.shape}")
print(f"Classification Validation: {X_val_clf.shape}")
print(f"\nClass balance (train):")
print(y_train_clf.value_counts(normalize=True))

### 4.2 Baseline Model Comparison (F1 Macro)

In [ ]:
models_clf = {
    "LightGBM (balanced)": LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    "Random Forest (balanced)": RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "LightGBM (is_unbalance)": LGBMClassifier(is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1),
}

results_clf = []
for name, model in models_clf.items():
    start = time.time()
    model.fit(X_train_clf, y_train_clf)
    pred = model.predict(X_val_clf)
    f1 = f1_score(y_val_clf, pred, average='macro')
    elapsed = time.time() - start
    results_clf.append({"Model": name, "F1 Macro": f1, "Time (s)": round(elapsed, 2)})
    print(f"{name:30s} → F1 Macro: {f1:.4f}  ({elapsed:.1f}s)")

display(pd.DataFrame(results_clf))

### 4.3 SMOTE Oversampling + LightGBM

We apply SMOTE to the **training set only** to create synthetic minority samples, then train and evaluate.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_clf, y_train_clf)

print(f"Before SMOTE: {y_train_clf.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}")

lgbm_smote = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1,
                             n_estimators=300, learning_rate=0.05, max_depth=7, num_leaves=31)
lgbm_smote.fit(X_train_sm, y_train_sm)
pred_smote = lgbm_smote.predict(X_val_clf)
f1_smote = f1_score(y_val_clf, pred_smote, average='macro')
print(f"\nLGBM + SMOTE F1 Macro: {f1_smote:.4f}")
print(classification_report(y_val_clf, pred_smote))

### 4.4 Classification Hyperparameter Tuning + Threshold Optimization

In [ ]:
clf_param_dist = {
    'n_estimators': stats.randint(100, 500),
    'learning_rate': stats.uniform(0.01, 0.14),
    'max_depth': [5, 7, 9, 11],
    'num_leaves': stats.randint(15, 80),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'min_child_samples': [5, 10, 20, 30, 50]
}

clf_search = RandomizedSearchCV(
    estimator=LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=clf_param_dist,
    n_iter=25,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    verbose=1
)

print("Tuning classifier on SMOTE data...")
clf_search.fit(X_train_sm, y_train_sm)

best_clf = clf_search.best_estimator_
print(f"\nBest params: {clf_search.best_params_}")

# Threshold optimization
proba = best_clf.predict_proba(X_val_clf)[:, 1]

best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.3, 0.7, 0.01):
    preds = (proba >= t).astype(int)
    f1_val = f1_score(y_val_clf, preds, average='macro')
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = t

print(f"\nOptimal threshold: {best_thresh:.2f}")
print(f"Best F1 Macro:     {best_f1:.4f}")
print(classification_report(y_val_clf, (proba >= best_thresh).astype(int)))

## 5. Final Pipeline — Generate Test Predictions

We retrain on the **full** training set and produce the output CSVs.

In [ ]:
start_time = time.time()

# === REGRESSION: retrain on full training data ===
X_full_reg = train_clean[base_features]
y_full_reg = train_clean['safety_rating']
X_test_reg = test_clean[base_features]

# Retrain best LightGBM
final_lgbm = best_lgbm
final_lgbm.fit(X_full_reg, y_full_reg)
pred_lgbm_final = final_lgbm.predict(X_test_reg)

# Retrain best CatBoost
final_cat = CatBoostRegressor(**best_cat_params, random_seed=42, verbose=0)
final_cat.fit(X_full_reg, y_full_reg)
pred_cat_final = final_cat.predict(X_test_reg)

# Ensemble
test_clean['safety_rating'] = best_w * pred_lgbm_final + (1 - best_w) * pred_cat_final

# === CLASSIFICATION: retrain on full SMOTE data ===
clf_features = base_features + ['safety_rating']
X_full_clf = train_clean[clf_features]
y_full_clf = train_clean['claim']

smote_full = SMOTE(random_state=42)
X_full_sm, y_full_sm = smote_full.fit_resample(X_full_clf, y_full_clf)

final_clf = best_clf
final_clf.fit(X_full_sm, y_full_sm)

X_test_clf = test_clean[clf_features]
test_proba = final_clf.predict_proba(X_test_clf)[:, 1]
test_clean['claim'] = (test_proba >= best_thresh).astype(int)

# === EXPORT ===
test_clean[['policy_id', 'safety_rating']].to_csv('z5543164_regression.csv', index=False)
test_clean[['policy_id', 'claim']].to_csv('z5543164_classification.csv', index=False)

total = time.time() - start_time
print(f"Predictions saved.")
print(f"Final pipeline runtime: {total:.2f}s")

# Evaluate against test ground truth (if available)
if 'safety_rating' in test_df.columns and test_df['safety_rating'].notna().all():
    rmse_test = root_mean_squared_error(test_df['safety_rating'], test_clean['safety_rating'])
    print(f"\n[EVAL] Test RMSE: {rmse_test:.4f}")
    if rmse_test <= 1.0:
        print("[EVAL] ✅ Regression: FULL MARKS")
    elif rmse_test < 10.0:
        print(f"[EVAL] Regression score: {(1 - (rmse_test - 1) / 9) * 5:.2f}/5")

if 'claim' in test_df.columns and test_df['claim'].notna().all():
    f1_test = f1_score(test_df['claim'], test_clean['claim'], average='macro')
    print(f"[EVAL] Test F1 Macro: {f1_test:.4f}")
    if f1_test >= 0.6:
        print("[EVAL] ✅ Classification: FULL MARKS")

## Summary

### Key Techniques Used

| Component | Technique |
|---|---|
| **Feature Parsing** | Regex extraction from `torque`, `power`, `car_age` |
| **Feature Indicators** | Binary columns for each safety feature in the `features` list |
| **Interactions** | power-to-weight, torque-to-weight, safety density, volume, etc. |
| **Encoding** | OrdinalEncoder with `unknown_value=-1` for unseen test categories |
| **Missing Values** | Training-set median for numerics; `'Missing'` for categoricals |
| **Regression** | LightGBM + CatBoost ensemble |
| **Classification** | SMOTE oversampling + LightGBM with threshold tuning |